# 01 — 数据概览 & 预处理

> 🎯 **适用场景**: 项目启动第一步——把数据接入环境，快速了解全貌
> ⏱️ **预计学习时长**: 45 分钟
> 📌 **核心问题**: 有多少张表？每张表长什么样？有没有缺失、重复、类型错误？

---

## 本 Notebook 结构

1. **数据导入** → 加载全部 9 张表，统一放在 `../data/` 下
2. **初次概览** → 每张表的 `shape` / `info` / `head`
3. **缺失值分析** → 可视化每一列的缺失比例
4. **重复值检查** → 主键是否唯一？有没有重复行？
5. **类型转换** → 日期解析 / 类别优化 / 数值校验
6. **数据质量总览** → 一张表汇总所有关键信息


In [ ]:
# ═══════════════════════════════════════════
# 0. 环境准备
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_DIR = "../data"

print("✅ 环境准备完成")

print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")


## 一、数据导入

📌 本项目包含 9 张表，数据模型如下：

```
olist_customers ──┐
                  ├── olist_orders ──┬── olist_order_items ── olist_products
                  │                  ├── olist_order_payments
                  │                  └── olist_order_reviews
olist_sellers ────┘
olist_geolocation (独立)
product_category_name_translation (翻译表)
```

核心表是 `olist_orders`（订单主表），通过 `order_id` 关联 items、payments、reviews。

In [ ]:
# ═══════════════════════════════════════════
# 1. 加载全部 9 张表
# ═══════════════════════════════════════════

# 列出 data 目录下所有文件，确保数据完整
data_files = sorted(os.listdir(DATA_DIR))
print("📁 data/ 目录文件列表:")
for f in data_files:
    size_mb = os.path.getsize(f"{DATA_DIR}/{f}") / (1024 * 1024)
    print(f"  {'✅' if f.endswith('.csv') else '⚠️'}  {f:45s}  {size_mb:7.2f} MB")

# ── 加载各表 ──
# 核心交易表
df_orders   = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv")
df_items    = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
df_payments = pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
df_reviews  = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")

# 维度表
df_customers = pd.read_csv(f"{DATA_DIR}/olist_customers_dataset.csv")
df_products  = pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
df_sellers   = pd.read_csv(f"{DATA_DIR}/olist_sellers_dataset.csv")

# 辅助表
df_geo    = pd.read_csv(f"{DATA_DIR}/olist_geolocation_dataset.csv")
df_trans  = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")

print("\n✅ 全部 9 张表加载完成")


## 二、初次概览

📌 快速查看每张表的基本信息：行数、列数、前几行。建立对数据的第一印象。

In [ ]:
# ═══════════════════════════════════════════
# 2. 每张表的 shape + 字段 + 示例
# ═══════════════════════════════════════════

tables = {
    '📦 olist_orders':               df_orders,
    '🛒 olist_order_items':           df_items,
    '💳 olist_order_payments':        df_payments,
    '⭐ olist_order_reviews':          df_reviews,
    '👤 olist_customers':             df_customers,
    '📐 olist_products':              df_products,
    '🏪 olist_sellers':               df_sellers,
    '📍 olist_geolocation':           df_geo,
    '🔤 product_category_translation': df_trans,
}

for name, df in tables.items():
    print(f"\n{'='*70}")
    print(f"{name}  »  {df.shape[0]:,} 行 × {df.shape[1]} 列")
    print(f"{'='*70}")
    print(f"  字段: {', '.join(df.columns[:8])}{' ...' if len(df.columns) > 8 else ''}")
    print(f"  示例 (前 2 行):")
    print(df.head(2).to_string(max_colwidth=30))


In [ ]:
# ── 每张表的 dtypes 一览 ──
print("📌 各表数据类型概览\n")

for name, df in tables.items():
    dtype_summary = df.dtypes.value_counts().to_dict()
    dtype_str = ', '.join([f"{v}×{k}" for k, v in dtype_summary.items()])
    print(f"  {name:40s}  {dtype_str}")


## 三、缺失值分析

📌 缺失值是数据质量的第一大问题。我们逐表、逐列检查缺失数量及比例，并通过热力图直观展示。

In [ ]:
# ═══════════════════════════════════════════
# 3. 缺失值统计 & 可视化
# ═══════════════════════════════════════════

def missing_report(df, name):
    """返回每列的缺失数、缺失率 DataFrame，并打印业务解读"""
    total = len(df)
    miss = df.isnull().sum()
    miss_pct = (miss / total * 100).round(2)

    report = pd.DataFrame({
        'column': miss.index,
        'missing': miss.values,
        'pct': miss_pct.values
    }).query('missing > 0').sort_values('missing', ascending=False)

    if len(report) > 0:
        print(f"\n📌 {name}:")
        print(report.to_string(index=False))
        print(f"   → 共 {len(report)}/{len(df.columns)} 列存在缺失值")
    else:
        print(f"  ✅ {name}: 无缺失值")
    return report

all_reports = {}
for name, df in tables.items():
    all_reports[name] = missing_report(df, name)


In [ ]:
# ── 缺失值热力图（所有表合并展示） ──
# 构建一个全量缺失矩阵：行=表名，列=字段名(去重)
all_miss = {}
for name, df in tables.items():
    short = name.replace('📦 ','').replace('🛒 ','').replace('💳 ','').replace('⭐ ','').replace('👤 ','').replace('📐 ','').replace('🏪 ','').replace('📍 ','').replace('🔤 ','')[:35]
    miss_rate = df.isnull().mean()
    all_miss[short] = miss_rate

df_miss = pd.DataFrame(all_miss).T * 100

# 只显示有缺失的列
has_miss_cols = df_miss.columns[df_miss.max() > 0]
df_miss = df_miss[has_miss_cols]

if len(has_miss_cols) > 0:
    fig, ax = plt.subplots(figsize=(14, max(4, len(all_miss) * 0.6)))
    sns.heatmap(df_miss, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=1, cbar_kws={'label': '缺失率 (%)'}, ax=ax,
                vmin=0, vmax=100)
    ax.set_title('各表各列缺失率热力图 (%)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/01_missing_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("✅ 所有表均无缺失值")


### 3.1 重点缺失字段解读

📌 并非所有缺失都是"坏"的——有些字段的缺失本身就有业务含义。

In [ ]:
# ── 订单表中的关键缺失分析 ──
# order_approved_at / order_delivered_carrier_date / order_delivered_customer_date
# 这些字段的缺失通常对应订单状态（如 cancelled）

print("📌 订单表时间字段缺失 vs 订单状态\n")

time_cols = ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']

for col in time_cols:
    missing_mask = df_orders[col].isnull()
    n_miss = missing_mask.sum()
    if n_miss > 0:
        print(f"  {col}: 缺失 {n_miss:,} 条 ({n_miss/len(df_orders)*100:.1f}%)")
        print(f"    → 对应订单状态分布:")
        status_dist = df_orders.loc[missing_mask, 'order_status'].value_counts()
        for status, cnt in status_dist.items():
            print(f"       {status}: {cnt:,} ({cnt/n_miss*100:.1f}%)")
        print()

# 📌 业务解读：订单被取消或未支付时，审批/发货/送达日期自然为空。
# 这属于"结构化缺失"，不需要填充，在分析时筛选 status='delivered' 即可。


## 四、重复值检查

📌 重复数据会导致指标虚高（如 GMV 翻倍）。我们检查：
1. 完全重复的行
2. 主键是否唯一

In [ ]:
# ═══════════════════════════════════════════
# 4. 重复值检查
# ═══════════════════════════════════════════

# ── 4.1 完全重复行 ──
print("📌 完全重复行检查\n")
for name, df in tables.items():
    dup_rows = df.duplicated().sum()
    short = name.split(' ')[-1][:35]
    if dup_rows > 0:
        print(f"  ⚠️  {short}: {dup_rows:,} 行完全重复")
    else:
        print(f"  ✅ {short}: 无重复行")


In [ ]:
# ── 4.2 主键唯一性检查 ──
print("\n📌 主键唯一性检查\n")

pk_checks = {
    'orders (order_id)':           ('order_id', df_orders),
    'customers (customer_id)':     ('customer_id', df_customers),
    'products (product_id)':       ('product_id', df_products),
    'sellers (seller_id)':         ('seller_id', df_sellers),
    'reviews (review_id)':         ('review_id', df_reviews),
}

for label, (pk, df) in pk_checks.items():
    total = len(df)
    unique = df[pk].nunique()
    dup_pk = df[pk].duplicated().sum()
    if dup_pk > 0:
        dup_ids = df[pk].value_counts()[df[pk].value_counts() > 1]
        print(f"  ⚠️  {label}: total={total:,}  unique={unique:,}  重复={dup_pk:,}")
        print(f"     重复最多的值: {dup_ids.head(3).to_dict()}")
    else:
        print(f"  ✅ {label}: total={total:,}  unique={unique:,}  主键唯一")

# ── 复合主键检查（items 表：order_id + order_item_id） ──
dup_items = df_items.duplicated(subset=['order_id', 'order_item_id']).sum()
print(f"\n  ✅ items (order_id+order_item_id): total={len(df_items):,}  复合主键重复={dup_items}")

# payments 表：order_id + payment_sequential
dup_pay = df_payments.duplicated(subset=['order_id', 'payment_sequential']).sum()
print(f"  ✅ payments (order_id+payment_sequential): total={len(df_payments):,}  复合主键重复={dup_pay}")


## 五、数据类型转换

📌 CSV 加载后默认类型可能不准确，我们需要：
1. 日期列 → `datetime64`（方便时间运算）
2. 类别列 → `category`（节省内存，加速分组）
3. 数值列校验（确保计算不出错）

In [ ]:
# ═══════════════════════════════════════════
# 5. 日期列解析 & 类型优化
# ═══════════════════════════════════════════

# ── 5.1 订单表日期列 ──
date_cols_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols_orders:
    if col in df_orders.columns:
        df_orders[col] = pd.to_datetime(df_orders[col])

# items 表
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'])

# reviews 表
for col in ['review_creation_date', 'review_answer_timestamp']:
    df_reviews[col] = pd.to_datetime(df_reviews[col])

print("✅ 日期列全部转换为 datetime64")

# ── 5.2 字段类型汇总 ──
print("\n📌 各表日期列时间范围:\n")
print(f"  orders.purchase_timestamp:  {df_orders['order_purchase_timestamp'].min()}  ~  {df_orders['order_purchase_timestamp'].max()}")
print(f"  orders.delivered_date:      {df_orders['order_delivered_customer_date'].min()}  ~  {df_orders['order_delivered_customer_date'].max()}")
print(f"  items.shipping_limit_date:  {df_items['shipping_limit_date'].min()}  ~  {df_items['shipping_limit_date'].max()}")
print(f"  reviews.creation_date:      {df_reviews['review_creation_date'].min()}  ~  {df_reviews['review_creation_date'].max()}")

# 📌 时间范围结论: 2016-09 ~ 2018-10，约 25 个月


In [ ]:
# ── 5.3 类别列优化为 category 类型 ──
cat_conversions = {
    'df_orders':   ['order_status'],
    'df_payments': ['payment_type'],
    'df_customers':['customer_state', 'customer_city'],
    'df_sellers':  ['seller_state', 'seller_city'],
    'df_products': ['product_category_name'],
}

for df_name, cols in cat_conversions.items():
    df_obj = globals()[df_name]
    for col in cols:
        if col in df_obj.columns:
            df_obj[col] = df_obj[col].astype('category')

print("✅ 类别列转换为 category 类型")

# ── 5.4 数值列校验 ──
# 检查 price / freight_value / payment_value 是否含负数或异常
num_checks = {
    'items.price':          (df_items, 'price'),
    'items.freight_value':  (df_items, 'freight_value'),
    'payments.payment_value': (df_payments, 'payment_value'),
    'payments.payment_installments': (df_payments, 'payment_installments'),
    'reviews.review_score':  (df_reviews, 'review_score'),
}

print("\n📌 数值列统计:\n")
for label, (df, col) in num_checks.items():
    vals = df[col]
    print(f"  {label:35s}  min={vals.min():>10.2f}  max={vals.max():>10.2f}  "
          f"mean={vals.mean():>10.2f}  median={vals.median():>10.2f}  "
          f"neg={ (vals<0).sum():>6,}  zero={ (vals==0).sum():>6,}")

# 📌 验证: price/freight 不应为负；review_score 应在 1~5


## 六、数据质量总览

📌 用一张大表汇总所有发现，方便后续快速定位问题。

In [ ]:
# ═══════════════════════════════════════════
# 6. 数据质量仪表盘
# ═══════════════════════════════════════════

def memory_mb(df):
    return df.memory_usage(deep=True).sum() / (1024 * 1024)

summary_rows = []
for name, df in tables.items():
    short = name.split(' ')[-1][:35]
    n_rows, n_cols = df.shape
    n_miss = df.isnull().sum().sum()
    miss_rate = n_miss / (n_rows * n_cols) * 100 if n_rows * n_cols > 0 else 0
    n_dup = df.duplicated().sum()
    mem = memory_mb(df)
    
    # 推断主键
    pk_candidates = [c for c in df.columns if c.endswith('_id') or c == 'zip_code_prefix']
    
    summary_rows.append({
        '表名': short,
        '行数': f'{n_rows:,}',
        '列数': n_cols,
        '缺失单元格': f'{n_miss:,}',
        '缺失率': f'{miss_rate:.1f}%',
        '重复行': f'{n_dup:,}',
        '内存(MB)': f'{mem:.1f}',
        '候选主键': ', '.join(pk_candidates[:3]),
    })

df_summary = pd.DataFrame(summary_rows)

fig, ax = plt.subplots(figsize=(16, 5))
ax.axis('off')
table = ax.table(cellText=df_summary.values,
                 colLabels=df_summary.columns,
                 cellLoc='center',
                 loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.6)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(weight='bold', color='white')
    elif row % 2 == 0:
        cell.set_facecolor('#ecf0f1')

ax.set_title('📊 数据质量总览仪表盘', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/01_data_quality_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# 也打印到控制台方便复制
print("\n📌 数据质量总览:\n")
print(df_summary.to_string(index=False))
print(f"\n📌 合计: {sum(len(df) for df in tables.values()):,} 行, "
      f"总内存 {sum(memory_mb(df) for df in tables.values()):.1f} MB")


## 📝 康奈尔笔记联动区

### 左侧栏（核心概念）
- **结构化缺失**: 订单取消导致时间字段为空 → 不是数据质量问题，而是业务状态反映
- **主键 vs 复合主键**: orders 用 order_id；items 用 order_id + order_item_id；payments 用 order_id + payment_sequential
- **类型优化**: `category` 类型可节省 60~80% 内存；`datetime64` 支持时间运算

### 右侧栏（反思问题）
> 💭 `order_estimated_delivery_date` 没有缺失——这个字段是下单时自动生成的，跟实际是否送达无关。
> 💭 `review_comment_title` 和 `review_comment_message` 缺失率很高，是用户懒得写还是有其他原因？
> 💭 geolocation 表有 100 万行却没有主键，意味着同一个 zip_code_prefix 可能有多条经纬度记录——该用哪个？

### 底部栏（行动清单）
- [ ] 确认已 parse 的日期列时间范围与 README 描述一致（2016-09 ~ 2018-10）
- [ ] 如果 geolocation 有多条同一邮编的记录，后续分析时决定聚合策略
- [ ] 到 `project01_reflections.md` 记录本 Notebook 发现的关键质量问题和应对策略
